# Sistema multiagente de soporte — Nimbus Cloud
**Proyecto Integrador M3 · AI Engineering (Henry)**

Un **Agente Orquestador** clasifica la intención de cada consulta (RR.HH., Tecnología, Finanzas o fuera de alcance) y la **delega de forma condicional** al **Agente RAG especializado** correcto, que responde *solo* con la documentación interna de su dominio. Todo el flujo está implementado con **LangChain** (LCEL) y **trazado completo con Langfuse**; como bonus, un **Agente Evaluador** puntúa cada respuesta de 1 a 10 con la Score API de Langfuse.

```
consulta → [Orquestador] → RunnableBranch → [HR | TECH | FINANCE | fallback] → respuesta → [Evaluador] → Score API
```

**Orden de ejecución:** correr las celdas de arriba hacia abajo (Secciones 1 → 6). Requiere `.env` con `OPENAI_API_KEY` y (opcional pero recomendado) las claves de Langfuse.

> La implementación vive en `src/` (módulos reutilizables y testeables); este notebook la importa, la explica y la ejecuta sección por sección.

## 1. Setup e imports
Se cargan las variables de entorno desde `.env` (sin secretos hardcodeados) y se verifica la configuración. Si faltan las claves de Langfuse el sistema funciona igual, pero **sin trazas**.

In [1]:
import json
import os
import sys
from pathlib import Path

# Permite importar `src` tanto si el notebook se abre desde la raíz como desde otra carpeta
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src import config, tracing
from src.knowledge_base import count_chunks_per_domain, load_domain_documents, split_documents

print("Modelo de chat      :", config.CHAT_MODEL)
print("Modelo de embeddings:", config.EMBEDDING_MODEL)
print("OpenAI configurado  :", bool(os.getenv("OPENAI_API_KEY")))
print("Langfuse configurado:", tracing.langfuse_enabled())

Modelo de chat      : gpt-4o-mini
Modelo de embeddings: text-embedding-3-small
OpenAI configurado  : True
Langfuse configurado: True


## 2. Carga de documentos y vector stores
Cada dominio tiene su **propia carpeta** (`data/hr_docs/`, `data/tech_docs/`, `data/finance_docs/`) y su **propia colección** en Chroma. Esto es el núcleo del enfoque multiagente: cada agente es experto en *su* base de conocimiento y no puede “contaminarse” con documentos de otras áreas.

**Decisiones de RAG**
- **Chunking:** `RecursiveCharacterTextSplitter` con `chunk_size=800` y `chunk_overlap=120` (~15 %). Corta primero por títulos y párrafos para no partir una política a la mitad; el solapamiento conserva contexto entre chunks vecinos.
- **Embeddings:** `text-embedding-3-small` (buen balance costo/calidad, mismo modelo para indexar y consultar).
- **Vector store:** Chroma persistido en `.vectorstores/` (se crea la primera vez y se reutiliza).
- **Retrieval:** `k=4` fragmentos por consulta: suficiente contexto sin inflar el prompt.

In [2]:
chunks_per_domain = count_chunks_per_domain()   # sin llamar a la API
for domain, n in chunks_per_domain.items():
    n_docs = len(load_domain_documents(domain))
    status = "OK" if n >= config.MIN_CHUNKS_PER_DOMAIN else "INSUFICIENTE"
    print(f"{domain:<8} {n_docs} documentos -> {n:>3} chunks  (mínimo exigido: {config.MIN_CHUNKS_PER_DOMAIN})  {status}")

HR       5 documentos -> 144 chunks  (mínimo exigido: 50)  OK
TECH     5 documentos -> 123 chunks  (mínimo exigido: 50)  OK
FINANCE  5 documentos -> 114 chunks  (mínimo exigido: 50)  OK


In [3]:
# Un ejemplo de chunk, para ver cómo queda la metadata
sample = split_documents(load_domain_documents("HR"))[3]
print(sample.metadata)
print(sample.page_content[:400], "...")

{'source': 'beneficios_compensaciones.md', 'domain': 'HR', 'chunk_id': 3}
## 2. Cobertura de salud

### 2.1 Obra social y prepaga

Todo el personal en Argentina recibe cobertura médica desde el primer día de contrato. Nimbus Cloud aporta el 100% de un plan prepago "Plan 310" para la persona titular y el 70% para el grupo familiar primario (cónyuge o conviviente e hijos hasta 25 años). La empresa trabaja con el prestador ficticio "Salud Integral Sur". Opciones:

- Plan 3 ...


## 3. Definición de agentes
`RAGAgent` (en `src/agents/rag_agent.py`) arma una cadena **LCEL**:

`{question} → retriever → formato de contexto → prompt de rol → LLM → parser`

Los tres especialistas (`hr_agent.py`, `tech_agent.py`, `finance_agent.py`) son instancias con su **vector store**, su **rol** (system prompt) y su **contacto de escalamiento**. El prompt les exige responder *solo* con el contexto y admitir cuando la documentación no alcanza (anti-alucinación). Cada paso lleva un `run_name` para que se lea bien en Langfuse.

In [4]:
from src.multi_agent_system import SupportSystem

# Construye vector stores (primera vez: calcula embeddings), agentes, orquestador y evaluador
system = SupportSystem()
print("Agentes especializados:", list(system.agents))
print({d: s._collection.count() for d, s in system.vector_stores.items()}, "chunks indexados")

Agentes especializados: ['HR', 'TECH', 'FINANCE']
{'HR': 144, 'TECH': 123, 'FINANCE': 114} chunks indexados


## 4. Orquestador y enrutamiento
El orquestador (`src/agents/orchestrator.py`) **no responde**: solo clasifica. Se eligió **salida estructurada** (`with_structured_output` + Pydantic) en vez de parsear texto libre, porque da una decisión validada y determinista (`department`, `sub_question`, `confidence`, `reasoning`).

**Patrones de L3 aplicados**
- **Roles claros y límites explícitos** en el prompt de clasificación.
- **Ruteo directo** (1 consulta → 1 agente) y **delegación en paralelo** (consultas con partes de varias áreas: cada sub-pregunta va a su agente y las respuestas se combinan).
- **Enrutamiento condicional** con `RunnableBranch` (HR / TECH / FINANCE / fallback).
- **Falla segura:** si la consulta no es de ningún área → `OUT_OF_SCOPE` (no se inventa una respuesta); si el orquestador falla, se registra el error y se devuelve el fallback.
- **Logging de cada delegación**: la decisión (`reasoning`, `confidence`) queda en la traza de Langfuse.

In [5]:
from src.agents.orchestrator import ORCHESTRATOR_SYSTEM_PROMPT
print(ORCHESTRATOR_SYSTEM_PROMPT)

Sos el Agente Orquestador del sistema de soporte interno de Nimbus Cloud.
Tu único trabajo es CLASIFICAR la intención de la consulta y decidir a qué agente especialista delegarla.
No respondas la consulta.

Áreas disponibles:
- HR: recursos humanos — vacaciones, licencias, beneficios, aguinaldo/bono, onboarding, evaluaciones de desempeño, carrera, trabajo remoto, conducta, renuncia/offboarding.
- TECH: soporte técnico/IT — contraseñas, cuentas, accesos, VPN, MFA, laptops y equipamiento, software y licencias, red/Wi-Fi, tickets y SLAs de IT, seguridad informática.
- FINANCE: finanzas — gastos y reembolsos, tarjetas corporativas, facturación a clientes, cobranzas, compras y proveedores, presupuesto, cierre contable, pagos y recibos de sueldo (aspecto contable/pago).
- OUT_OF_SCOPE: cualquier cosa que no encaje en las áreas anteriores (clima, recetas, política, chistes, conocimiento general).

Reglas:
1. Elegí el área por el TEMA principal de la consulta, no por palabras sueltas (ej.: "re

In [6]:
# Clasificación aislada (sin ejecutar los agentes)
for q in [
    "¿Cómo pido una licencia por estudio?",
    "Mi VPN no conecta y además quiero saber cuándo cobro el aguinaldo",
    "¿Quién ganó el mundial de 2022?",
]:
    d = system.classify(q)
    print(q)
    print("   ->", [(r.department, r.sub_question) for r in d.routes], f"conf={d.confidence:.2f}")
    print("   ", d.reasoning)

¿Cómo pido una licencia por estudio?
   -> [('HR', '¿Cómo pido una licencia por estudio?')] conf=0.90
    La consulta se refiere a un trámite relacionado con recursos humanos, específicamente sobre licencias.


Mi VPN no conecta y además quiero saber cuándo cobro el aguinaldo
   -> [('TECH', 'Mi VPN no conecta.'), ('HR', '¿Cuándo cobro el aguinaldo?')] conf=0.80
    La consulta incluye un problema técnico relacionado con la VPN, que corresponde al área de soporte técnico, y una pregunta sobre el aguinaldo, que es un tema de recursos humanos.


¿Quién ganó el mundial de 2022?
   -> [('OUT_OF_SCOPE', '¿Quién ganó el mundial de 2022?')] conf=1.00
    La consulta es sobre un evento deportivo y no se relaciona con ninguna de las áreas disponibles.


## 5. Pruebas y ejemplos
Se ejecutan las **14 consultas de `test_queries.json`** (el *golden dataset* del proyecto: todas las áreas + casos borde — multi-dominio, ambigua, en inglés, vaga y fuera de alcance), cada una con su `expected_intent`. Para casos ambiguos hay `acceptable_intents`. Toda la corrida lleva el mismo tag (`golden-run-…`) para filtrarla junta en Langfuse y comparar versiones.

In [7]:
queries = json.loads(config.TEST_QUERIES_PATH.read_text(encoding="utf-8"))
print(len(queries), "consultas de prueba;",
      "categorías esperadas:", sorted({d for q in queries for d in q["expected_intent"].split("+")}))
for q in queries:
    print(f"#{q['id']:>2} [{q['expected_intent']:<13}] {q['type']}")

14 consultas de prueba; categorías esperadas: ['finance', 'hr', 'out_of_scope', 'tech']
# 1 [hr           ] directa
# 2 [hr           ] directa
# 3 [tech         ] directa
# 4 [tech         ] directa
# 5 [finance      ] directa
# 6 [finance      ] directa
# 7 [tech         ] directa
# 8 [hr           ] directa
# 9 [finance+tech ] caso borde: multi-dominio (delegación en paralelo)
#10 [hr           ] caso borde: tema de RR.HH. con palabras de portal/IT
#11 [finance      ] caso borde: ambigua (FINANCE o HR)
#12 [out_of_scope ] caso borde: fuera de alcance
#13 [tech         ] caso borde: consulta en inglés
#14 [out_of_scope ] caso borde: consulta vaga / mal formada


In [8]:
from run_tests import accepted_intents

RUN_TAG = "golden-run-notebook-v1"   # un mismo tag para toda la corrida -> se filtra junta en Langfuse
results, correct = [], 0
for q in queries:
    out = system.answer(q["query"], session_id=RUN_TAG, user_id="notebook",
                        expected_intent=q["expected_intent"].upper().split("+"), run_tag=RUN_TAG)
    chosen = out["routing"]["departments"]
    ok = all(d in accepted_intents(q) for d in chosen)
    correct += ok
    score = out["evaluation"]["overall"] if out["evaluation"] else "-"
    print(f"[{'OK ' if ok else 'ERR'}] #{q['id']:>2} esperado={q['expected_intent']} elegido={chosen} conf={out['routing']['confidence']:.2f} calidad={score}")
    print(f"        {q['query']}")
    results.append({"test": q, "ok": ok, **out})
print(f"\nRouting correcto: {correct}/{len(queries)} ({correct/len(queries):.0%})")

[OK ] # 1 esperado=hr elegido=['HR'] conf=0.90 calidad=10
        ¿Cuántos días de vacaciones me corresponden después de un año de antigüedad?


[OK ] # 2 esperado=hr elegido=['HR'] conf=0.90 calidad=10
        ¿Cómo solicito una licencia por enfermedad y qué certificado tengo que presentar?


[OK ] # 3 esperado=tech elegido=['TECH'] conf=0.90 calidad=10
        No puedo conectarme a la VPN desde mi casa, ¿qué hago?


[OK ] # 4 esperado=tech elegido=['TECH'] conf=0.90 calidad=9
        ¿Cómo reseteo mi contraseña de Okta si perdí el acceso al MFA?


[OK ] # 5 esperado=finance elegido=['FINANCE'] conf=0.90 calidad=9
        ¿Cuál es el plazo para que me reembolsen un gasto de viaje y qué comprobantes necesito?


...

[OK ] #10 esperado=hr elegido=['HR'] conf=0.80 calidad=9
        ¿Cuántos días de duelo me corresponden y cómo cargo el pedido en el portal?


[OK ] #11 esperado=finance elegido=['FINANCE'] conf=0.90 calidad=7
        ¿Cómo se reintegra el gasto de internet del home office?


[OK ] #12 esperado=out_of_scope elegido=['OUT_OF_SCOPE'] conf=1.00 calidad=-
        ¿Cuál es la receta de la milanesa napolitana?


[OK ] #13 esperado=tech elegido=['TECH'] conf=0.90 calidad=10
        How do I request a software license for Figma?


[OK ] #14 esperado=out_of_scope elegido=['OUT_OF_SCOPE'] conf=0.50 calidad=-
        ayuda

Routing correcto: 14/14 (100%)


In [9]:
# Ejemplo completo de una consulta multi-dominio (delegación en paralelo)
multi = next(r for r in results if r["test"]["id"] == 9)
print("PREGUNTA:", multi["question"])
print("AGENTES :", multi["agents_used"])
print("RESPUESTA:\n", multi["answer"])

PREGUNTA: Necesito saber cuándo se pagan los sueldos y además mi VPN no conecta.
AGENTES : ['finance-agent', 'tech-agent']
RESPUESTA:
 **Finanzas**
Los sueldos se acreditan el último día hábil del mes. Si ese día es feriado o fin de semana, el pago se adelanta al día hábil anterior.

**Soporte Técnico**
Para diagnosticar por qué tu VPN no conecta, sigue estos pasos:

1. **Verificá tu Internet**: Asegúrate de que puedes abrir `https://www.google.com`. Si no puedes, el problema es tu conexión y no de la VPN.
2. **Cerrá y reabrí el cliente** de GlobalProtect, asegurándote de que el portal escrito sea exactamente `vpn.nimbuscloud.example`.
3. **Probá el MFA**: Si no recibís la notificación de Okta Verify, abre la app manualmente y utiliza el código TOTP.
4. **Cambiá de red**: Intenta conectarte usando la red de tu celular (hotspot), ya que algunos routers pueden bloquear el puerto UDP 4501.
5. **Usá el modo SSL**: En el cliente, ve a *Ajustes > Conexión > Forzar SSL (TCP 443)*.
6. **Reinic

## 6. Integración con Langfuse (observability) y Agente Evaluador (bonus)
**Tracing.** `src/tracing.py` abre **una traza raíz por consulta** (`support-query`, con `session_id`, `user_id` y tags) y entrega un `CallbackHandler` de LangChain. Con ese handler, *todo* lo que ocurre adentro queda anidado en la misma traza: la clasificación del orquestador, el `RunnableBranch`, el retriever de cada agente (con los chunks recuperados), cada llamada al LLM (tokens, latencia) y el juez del evaluador. Si una pregunta se enruta mal, se abre la traza y se ve **qué decidió el orquestador y por qué**; si una respuesta es mala, se ve **qué recuperó el retriever**.

**Evaluador.** `src/evaluator.py` es un LLM-as-judge que recibe *consulta + contexto recuperado + respuesta* y devuelve `relevance`, `completeness`, `accuracy` y un `overall` (1-10) con justificación. Los puntajes se registran en la traza con la **Score API** (`langfuse.create_score(...)`), por lo que se pueden filtrar y graficar en Langfuse (Scores / Dashboards) para detectar respuestas de baja calidad.

In [10]:
scored = [r for r in results if r["evaluation"]]
print(f"{len(scored)} respuestas RAG evaluadas")
print(f"Calidad promedio (overall): {sum(r['evaluation']['overall'] for r in scored)/len(scored):.1f} / 10")
low = [r for r in scored if r["evaluation"]["low_quality"]]
print(f"Respuestas marcadas como baja calidad (<6): {len(low)}")
for r in low:
    print(" -", r["question"], "->", r["evaluation"]["overall"], "|", r["evaluation"]["reason"][:160])

12 respuestas RAG evaluadas
Calidad promedio (overall): 9.2 / 10
Respuestas marcadas como baja calidad (<6): 0


In [11]:
# Enlaces a las trazas en Langfuse (una por consulta)
for r in results[:5]:
    print(r["test"]["id"], r.get("trace_url"))

1 https://cloud.langfuse.com/project/cmv5e4dm9033cad0dcnixtqxh/traces/fd0ce241cd94618d977f78c4a489febc
2 https://cloud.langfuse.com/project/cmv5e4dm9033cad0dcnixtqxh/traces/960a10accbae0fe488e4879c99f74a21
3 https://cloud.langfuse.com/project/cmv5e4dm9033cad0dcnixtqxh/traces/34a475a1a584b38ca156505006f9aa4c
4 https://cloud.langfuse.com/project/cmv5e4dm9033cad0dcnixtqxh/traces/6186c4dd2b3046b7021954aad2456ad2
5 https://cloud.langfuse.com/project/cmv5e4dm9033cad0dcnixtqxh/traces/4cc239b6c9fb513083ff8581dc50ee4e


In [12]:
# Guardar el detalle de la corrida y enviar los eventos pendientes a Langfuse
config.OUTPUTS_DIR.mkdir(exist_ok=True)
(config.OUTPUTS_DIR / "notebook_results.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
tracing.flush()
print("Listo: outputs/notebook_results.json guardado y trazas enviadas.")

Listo: outputs/notebook_results.json guardado y trazas enviadas.


## Limitaciones conocidas
- El corpus es **estático y ficticio** (Nimbus Cloud); en producción habría que reindexar ante cada cambio de documentos.
- El orquestador es un **clasificador por prompt**: es muy bueno con temas claros, pero en consultas realmente ambiguas puede elegir un área distinta de la esperada (por eso se registra la `confidence`).
- No hay **memoria conversacional** entre turnos: cada consulta es independiente (el `session_id` agrupa trazas en Langfuse, pero los agentes no comparten estado).
- El evaluador usa el mismo modelo que los agentes; para evaluación crítica convendría un modelo juez distinto o revisión humana en una muestra.